# Turkish Medical Agentic RAG

Run this notebook from the `agentic_rag/` directory. It validates the corpus/index, connects Together AI, and exposes a stateful chat helper.

In [ ]:
from dataclasses import asdict
from statistics import median

from IPython.display import Markdown, display
from langchain_core.messages import HumanMessage
from transformers import AutoTokenizer

from scripts.indexing.chunker import ChunkingConfig, chunk_parent_child_documents
from scripts.indexing.indexer import ensure_index
from scripts.indexing.loader import load_markdown_documents
from scripts.llm.together import create_together_llms
from scripts.retrieval.retriever import load_retriever
from scripts.settings import RagSettings
from scripts.workflow.graph import build_graph

settings = RagSettings.from_env()
settings

## Preview the parent–child chunks

The preview chunks one document only; the index step processes all documents.

In [ ]:
documents = load_markdown_documents(settings.data_dir)
tokenizer = AutoTokenizer.from_pretrained(settings.embedding_model)
parent_config = ChunkingConfig(
    tokenizer_name=settings.embedding_model,
    max_tokens=settings.parent_chunk_size_tokens,
    overlap_tokens=0,
)
child_config = ChunkingConfig(
    tokenizer_name=settings.embedding_model,
    max_tokens=settings.chunk_size_tokens,
    overlap_tokens=settings.chunk_overlap_tokens,
)
preview_parents, preview_children = chunk_parent_child_documents(
    documents[:1], parent_config, child_config, tokenizer=tokenizer
)
parent_token_counts = [len(tokenizer.encode(parent.page_content, add_special_tokens=False)) for parent in preview_parents]
child_token_counts = [len(tokenizer.encode(child.page_content, add_special_tokens=False)) for child in preview_children]
{
    'documents': len(documents),
    'preview_source': documents[0].metadata['source'],
    'preview_parents': len(preview_parents),
    'preview_children': len(preview_children),
    'parent_median_tokens': median(parent_token_counts),
    'parent_max_tokens': max(parent_token_counts),
    'child_median_tokens': median(child_token_counts),
    'child_max_tokens': max(child_token_counts),
}

## Create or validate Qdrant and the local parent store

Set `FORCE_REINDEX = True` once to migrate an older index. Set it back to `False` after rebuilding.

In [ ]:
FORCE_REINDEX = False
index_report = ensure_index(settings, force_recreate=FORCE_REINDEX)
asdict(index_report)

In [4]:
question = """

Laringeal ayna ile indirekt laringoskopi yapılan bir
hastada işlem sırasında aşağıdakilerden hangisinin
görülmesi en az olasıdır?
A) Plica aryepiglottica
B) Tuberculum corniculatum
C) Plica vestibularis
D) Incisura interarytenoidea
E) Tuberculum thyroideum superius

"""

In [ ]:
retriever = load_retriever(settings)
sanity_results = retriever.invoke(question)
[
    {
        'source': document.metadata.get('source'),
        'title': document.metadata.get('title'),
        'subtitle': document.metadata.get('subtitle'),
        'chunk_index': document.metadata.get('chunk_index'),
    }
    for document in sanity_results
]

## Build the Together AI graph

In [6]:
llms = create_together_llms(settings)
graph = build_graph(llms.responder, llms.pruning, retriever)
display(Markdown(f"```mermaid\n{graph.get_graph().draw_mermaid()}\n```"))

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	responder(responder)
	retrieval_tools(retrieval_tools)
	prune(prune)
	compact(compact)
	__end__([<p>__end__</p>]):::last
	__start__ --> responder;
	prune --> responder;
	responder -.-> compact;
	responder -.-> retrieval_tools;
	retrieval_tools --> prune;
	compact --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [7]:
conversation = {'messages': []}

def chat(user_input: str) -> str:
    if not user_input.strip():
        raise ValueError('user_input cannot be empty')
    conversation['messages'].append(HumanMessage(content=user_input.strip()))
    result = graph.invoke(conversation)
    conversation['messages'] = result['messages']

    for message in result['messages']:
        print(message)
        print()
    return result['messages'][-1].content


In [8]:
question = """

Laringeal ayna ile indirekt laringoskopi yapılan bir
hastada işlem sırasında aşağıdakilerden hangisinin
görülmesi en az olasıdır?
A) Plica aryepiglottica
B) Tuberculum corniculatum
C) Plica vestibularis
D) Incisura interarytenoidea
E) Tuberculum thyroideum superius

"""

In [10]:
answer = chat("bu sorunun cevabı retrieve edilen datada var mıydı yoksa kendi bilgini mi kullandın?")
display(Markdown(answer))

content='Laringeal ayna ile indirekt laringoskopi yapılan bir\nhastada işlem sırasında aşağıdakilerden hangisinin\ngörülmesi en az olasıdır?\nA) Plica aryepiglottica\nB) Tuberculum corniculatum\nC) Plica vestibularis\nD) Incisura interarytenoidea\nE) Tuberculum thyroideum superius' additional_kwargs={} response_metadata={} id='f12ccb15-3fdc-4d01-887d-c65effb1cbdb'

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 429, 'prompt_tokens': 618, 'total_tokens': 1047, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 395, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'zai-org/GLM-5.3-Flash', 'system_fingerprint': 'default', 'id': 'c90d2fd8-aws_ec1', 'finish_reason': 'tool_calls', 'logprobs': None} name='responder' id='lc_run--01a0d4e4-c68a-76d1-b16b-92be

Dürüst cevap: **büyük ölçüde kendi bilgimi kullandım.** Retrieve edilen doküman sorunun cevabını içermiyordu.

Getirilen belgede (ANATOMİ/ANATOMİ 1.pdf.md, chunk 429) yalnızca şunlar vardı:

- **Plica vestibularis**'in larinksin iç yapısına ait olduğu ve ventriculus laryngis'in sınırını oluşturduğu bilgisi → cevabın yalnızca küçük bir parçasını (C seçeneğinin larinks içi bir yapı olduğunu) dolaylı olarak destekledi.

Buna karşılık cevabı belirleyen asıl bilgiler retrieve edilen datada **yoktu**:

- İndirekt laringoskopide hangi yapıların görülüp görülemeyeceği (belgenin kendisi de "indirekt laringoskopi tekniği tanımlanmamıştır" notuyla bunu açıkça belirtiyordu)
- Plica aryepiglottica, tuberculum corniculatum, incisura interarytenoidea ve tuberculum thyroideum superius ile ilgili hiçbir bilgi
- Tuberculum thyroideum superius'un cartilago thyroidea'nın dış yüzeyinde olduğu bilgisi

Yani **E şıkkı (tuberculum thyroideum superius)** ve laringoskopik görünürlük mantığı tamamen kendi anatomik bilgimden geldi. Bunu önceki cevabımın sonundaki "Not" bölümünde kısmen belirtmiştim, ancak şimdi daha net itiraf ediyorum: cevap retrieve edilen kanıtlara dayanmıyor, model bilgisine dayanıyor. Bu, RAG sisteminin amacı açısından bir eksiklik — sorunun cevabını içeren bir doküman (ör. laringoskopi tekniği veya larinksin dış/iç yüzey anatomisini karşılaştıran bir kaynak) veri tabanında yok ya da sorgu onu getiremedi.